# [ RENPLAY AI ] - PIPELINE OFICIAL NO GOOGLE COLAB
### Motor de Traducao e Localizacao para Jogos Ren'Py com Qwen3.5-4B (Unsloth FP16)
---
**Funcionalidades:**
- Download direto de jogos via **Pixeldrain** (`pixeldrain.com/u/...`)
- Fila dinamica de processamento: adicione novos links enquanto a IA trabalha
- Protecao deterministica de tags Ren'Py (`{b}`, `{color}`, `[player_name]`)
- Memoria de traducao em SQLite e aprendizado continuo
- **Geracao de Arquivo Unico:** Patch `.rpy` com tela de idiomas integrada nas opcoes do jogo


## 1. Verificacao de Hardware & GPU
Verifica se a GPU do Colab esta ativa (recomendada: NVIDIA T4 16GB com precisao FP16).

In [ ]:
!nvidia-smi
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma'
print(f'[INFO] GPU Detectada: {gpu_name}')
if 'T4' in gpu_name:
    print('[OK] GPU Tesla T4 (Turing) detectada. Precisao ideal: torch.float16')


## 2. Montar Google Drive & Clonar Repositorio
Monta o Google Drive para salvar patches, checkpoints e o banco de dados de memoria permanentemente.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

# Pasta permanente no Google Drive
DRIVE_DIR = Path('/content/drive/MyDrive/RenplayAI')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'patches').mkdir(exist_ok=True)
(DRIVE_DIR / 'checkpoints').mkdir(exist_ok=True)
(DRIVE_DIR / 'data').mkdir(exist_ok=True)

# Clonar ou atualizar o repositorio da Renplay AI
REPO_URL = 'https://github.com/Ricardosvyt/ia-renpy.git' # Substitua pela URL do seu repositorio no GitHub
LOCAL_REPO = Path('/content/renplay-ai')

if not LOCAL_REPO.exists():
    print('[INFO] Clonando repositorio Renplay AI...')
    !git clone {REPO_URL} /content/renplay-ai
else:
    print('[INFO] Repositorio ja presente. Atualizando...')
    !cd /content/renplay-ai && git pull

sys.path.insert(0, str(LOCAL_REPO))


## 3. Instalacao de Dependencias & Unsloth
Instala as bibliotecas de processamento e a biblioteca Unsloth otimizada.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" triton
!pip install requests tqdm rich pydantic py7zr


## 4. Gerenciamento da Fila de Jogos
Adicione links do **Pixeldrain** a qualquer momento. Voce pode rodar esta celula sempre que quiser colocar um novo jogo na fila.

In [ ]:
from core.queue_manager import QueueManager

# Sincroniza a fila diretamente com o Google Drive para nao perder nenhum item
queue_path = DRIVE_DIR / 'data' / 'fila_jogos.json'
queue_mgr = QueueManager(queue_file_path=queue_path)

# --- ADICIONE SEUS LINKS AQUI ---
# Exemplo: informe a URL do Pixeldrain e o nome do jogo
# queue_mgr.add_game(url='https://pixeldrain.com/u/EXEMPLO', name='NomeDoJogo', target_languages=['pt-BR'])

print('[FILA ATUAL]')
for item in queue_mgr.list_queue():
    print(f"- [{item['status'].upper()}] {item['name']} ({item['url']})")


## 5. Carregar Modelo Qwen3.5-4B (FP16)
Carrega o Qwen3.5-4B em precisao 16-bit estavel sem quantizacao 4-bit, otimizado para a GPU T4.

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = 'unsloth/Qwen3.5-4B'
max_seq_length = 2048

print(f'[INFO] Carregando {model_name} em FP16...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=torch.float16,
)
FastLanguageModel.for_inference(model)
print('[OK] Modelo carregado e pronto para traducao!')


## 6. Execucao do Pipeline: Download, Extracao & Traducao
Processa o proximo jogo da fila, gera o **Patch de Arquivo Unico** e salva no Google Drive.

In [ ]:
from core.pixeldrain_downloader import PixeldrainDownloader
from core.renpy_extractor import RenpyExtractor
from core.tag_protector import TagProtector
from core.sqlite_memory import SQLiteMemory
from core.patch_builder import PatchBuilder

memory = SQLiteMemory(db_path=DRIVE_DIR / 'data' / 'memoria_traducoes.sqlite')
protector = TagProtector()
downloader = PixeldrainDownloader()
patch_builder = PatchBuilder(output_dir=DRIVE_DIR / 'patches')

item = queue_mgr.get_next_pending()
if not item:
    print('[INFO] Nenhum jogo pendente na fila. Adicione links na celula 4!')
else:
    print(f"[PROCESSANDO] {item['name']} ({item['url']})")
    queue_mgr.update_status(item['id'], 'downloading')
    
    try:
        # 1. Download do Pixeldrain
        archive = downloader.download(item['url'])
        
        # 2. Extracao do Jogo
        queue_mgr.update_status(item['id'], 'extracting')
        game_root = downloader.extract_game(archive)
        
        # 3. Extracao de textos
        queue_mgr.update_status(item['id'], 'translating')
        extractor = RenpyExtractor(game_root)
        units = extractor.extract_all()
        print(f'[INFO] Total de unidades extraidas: {len(units)}')
        
        # 4. Traducao com Protecao de Tags e Memoria SQLite
        translations = []
        target_lang = item['target_languages'][0] if item['target_languages'] else 'pt-BR'
        
        for unit in units:
            orig = unit['original_text']
            char = unit.get('character', 'Narrador')
            
            # Consultar memoria SQLite antes de acionar o modelo
            cached = memory.get_translation(orig, target_lang=target_lang, character=char)
            if cached:
                translations.append({'original_text': orig, 'translated_text': cached})
                continue
            
            # Proteger tags e variaveis
            masked_text, mask_map = protector.protect(orig)
            
            # Prompt de traducao para Qwen3.5-4B
            prompt = f"<|im_start|>system\nVoce e a Renplay AI. Traduza o dialogo para Portugues do Brasil preservando todas as mascaras __TAG_X__ e __VAR_X__ intactas.<|im_end|>\n<|im_start|>user\nPersonagem: {char}\nTexto: {masked_text}<|im_end|>\n<|im_start|>assistant\n"
            inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
            outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.3, use_cache=True)
            raw_resp = tokenizer.batch_decode(outputs)[0]
            pred = raw_resp.split('<|im_start|>assistant\n')[-1].replace('<|im_end|>', '').strip()
            
            # Restaurar tags e variaveis originais
            final_text = protector.restore(pred, mask_map)
            
            # Salvar na memoria SQLite
            memory.save_translation(orig, final_text, target_lang=target_lang, character=char, game_name=item['name'])
            translations.append({'original_text': orig, 'translated_text': final_text})
        
        # 5. Gerar Patch de Arquivo Unico com Seletor de Idiomas
        patch_file = patch_builder.build_single_file_patch(
            game_name=item['name'],
            translations=translations,
            target_lang=target_lang,
        )
        
        queue_mgr.update_status(item['id'], 'completed', patch_path=str(patch_file))
        print(f'\n[SUCESSO] Traducao finalizada! Patch salvo no Google Drive:\n{patch_file}')
        print('Instrucao: Basta copiar este unico arquivo para a pasta game/ do jogo!')
        
    except Exception as e:
        print(f'[ERRO] Falha ao processar jogo: {e}')
        queue_mgr.update_status(item['id'], 'error', error_message=str(e))
